# Notebook 1: Data Preparation & Baseline CNN

## Overview

This notebook covers the **data preparation pipeline** for our ABC Sign Language Recognition project:

1. **Understanding the data structure** - How images are labeled using directory structure
2. **Group-aware data splitting** - Preventing data leakage by splitting by person (not by image)
3. **Processing images** into TensorFlow Dataset objects for efficient training
4. **Training a baseline CNN** from scratch to establish a performance benchmark

### Why Group-Aware Splitting?

In typical image classification, we randomly split images into train/val/test sets. But here, the same person's hand appears in multiple photos. If we split randomly, the model could learn to recognize *specific hands* rather than *sign language gestures*. Group-aware splitting ensures that **all photos from one person stay in the same split**, forcing the model to generalize to entirely new people.

### Dataset

- **Classes:** A, B, C (German Sign Language / DGS letters)
- **Images:** ~884 photos from 24 different individuals
- **Format:** JPG images of varying resolutions, resized to 256x256

---

In [ ]:
import os
import shutil
from typing import TYPE_CHECKING

import numpy as np
import tensorflow as tf
from tensorflow import Tensor, concat
from tensorflow.data import AUTOTUNE, Dataset
from tensorflow.keras.callbacks import EarlyStopping, History
from tensorflow.keras.layers import (
    Conv2D,
    Dense,
    Flatten,
    Input,
    MaxPooling2D,
    Rescaling,
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import image_dataset_from_directory

if TYPE_CHECKING:
    from numpy.random import Generator

---
## 1. Labeling Image Data  🏷

Images are different from the kind of tabular data we've worked with before. It's not as obvious how we can tell tensorflow what class the image is when we can't have a column for the the label. Tensorflow has a function `image_dataset_from_directory` that is built to make this easier for us. All we have to do it put the images in folders with their class names and the function will automatically pair the photo (`X`) with the name of the folder it's in (`y`) when creating the `tensorflow.data.Dataset` object.

The directory structure that we provide to `image_dataset_from_directory` should look something like this:
```
hands_photos/
  ├── A/
  │    ├── image.png            
  │    ├── other_image.jpg  <--- you can mix file types, tensorflow is very flexible  
  │    └── ...
  ├── B/
  └── C/
```
This is especially helpful because we don't have to worry about the filenames of the images or anything like that, we just have to place them in the proper folders.


## 2. Grouping the Data  🗃

As explained on the LMS, we need to do group-aware splitting for this project. If we shuffled everyone's images together, then the training, validation, and testing sets will have different *pictures*, but they'll be the same *people*'s hands in all of the sets. We want to know if the model can generalize to new *people*'s hands, not just new photos of the same people. This requires group-aware data splitting where each person is considered a group and we split the data entire groups at a time. This way we don't spread any groups across multiple sets and the model has to generalize to entirely new people during evaluation.

Just like the the labels, indicating this metadata about the image (what group it is in) can be done with the file structure. All we have to do is give each group it's own folder:
```
data/
  └── grouped/
       ├── ben/           <--- each group (person) gets their own folder
       │    ├── A/        <--- images are still sorted by their label
       │    ├── B/
       │    └── C/
       ├── rockwell/
       │    ├── A/
       │    ├── B/
       │    └── C/
       ├── vasil/
       │    ├── A/
       │    ├── B/
       │    └── C/
       └── ...
```
Your dataset should currently be organized this way on your local machine.

## 3. Downloading Data 📥

Download the zip file which your instructor shared with you. You'll want to unzip it before placing it in your project's root folder. Use the following file structure:
```
project_root/             <--- you should change this to a more meaningful name
  ├── notebooks/
  │    └── this_notebook.ipynb
  └── data/               
       └── grouped/       <--- place data inside unziped folder here
```

---
## 4. Group-Aware Data Splitting  👁

Now that we have the grouped data, we're ready to split it. We can take advantage of the directory structure to determine what groups there are. Once we've determined what groups we have, we can randomly select some groups for each split and gather their images together. After the splitting, we'll want the data organized like this:
```
data/
  └── raw/             <--- we're calling it "raw/" because later we will store a preprocessed version of the data in a "processed/" folder
       ├── train/      <--- data is now split into train, test, val instead of being organized by group
       │    ├── A/     <--- we still have the data sorted by label
       │    ├── B/
       │    └── C/
       ├── test/
       │    ├── A/
       │    ├── B/
       │    └── C/
       └── val/
            ├── A/
            ├── B/
            └── C/
```
### Note on Independent Random Seeds:
While we want to keep the testing set fixed, it's not uncommon in deep learning to occassionally change which data is in training and which is in validation since we can't afford to do systematic cross-validation. To make this possible, we need two separate random seeds; one for determining which of the groups end up in the testing set (which we shouldn't change), and one for determining which of the remaining groups end up in the validation set (which we can try varying to check that performance is stable across fits).

### Note on Symlinks:
The code below will randomly split the grouped data and add this new file tree to the `data/` folder. Unless you are on windows, it will do this using something called "symlinks" to avoid duplicating all the images and taking up twice as much space. Instead, the files in the `raw/` folder simply point to the files in the `grouped/` folder. **This means you cannot delete the `grouped/` folder or you'll lose the images!**

In [ ]:
def _monte_carlo_split(
    groups_dict: dict[str, int],
    target_samples: int,
    seed: int,
    iterations: int = 1000,
) -> tuple[list[str], list[str]]:
    """
    Helper function for group_aware_split. Method for randomized splitting of variously sized groups according to a target sample size.
    We might want x% of the data in the test set, but if groups are varying sizes we can't rely on x% of groups equaling x% of the total samples.
    The monte carlo method repeatedly shuffles the groups and tries to split them as close to the target ratio as possible, and then returns the closest split it found.

    Args:
        groups_dict (dict): Mapping of group names to their total number of images.
        target_samples (int): The number of samples desired for the split. Algorithm will get as close as it can, without spreading any groups across splits.
        seed (int): Random seed for reproducibility.
        iterations (int): Number of random shuffles to try.

    Returns:
       tuple[list[str], list[str]]: List of group names in the split and list of remaining group names.
    """
    # Don't split if split size is 0
    if target_samples == 0:
        return [], list(groups_dict.keys())

    group_names: list[str] = list(groups_dict.keys())

    # Initialize variables to store whatever split is the best so far
    # Whenever the algorithm finds something better, it will overwrite what is stored here
    best_error: float = float('inf')
    best_split: list[str] = []
    best_remaining: list[str] = []

    for i in range(iterations):
        # Randomly shuffle groups
        iteration_rng: Generator = np.random.default_rng(seed + i)
        shuffled_groups: list[str] = iteration_rng.permutation(group_names).tolist()

        # Calculate the cumulative sum of samples for this specific shuffle
        cum_sums: np.ndarray = np.cumsum([groups_dict[g] for g in shuffled_groups])

        # Find which index gets us closest to our target sample count
        errors: np.ndarray = np.abs(cum_sums - target_samples)
        best_idx_for_this_shuffle = np.argmin(errors)

        # If this shuffle produced a more accurate split than previous ones, save it
        if errors[best_idx_for_this_shuffle] < best_error:
            best_error: int = errors[best_idx_for_this_shuffle]
            best_split = shuffled_groups[:best_idx_for_this_shuffle + 1]
            best_remaining = shuffled_groups[best_idx_for_this_shuffle + 1:]

            # Early stopping if we hit it exactly
            if best_error == 0:
                break

    return best_split, best_remaining


def group_aware_split(
    grouped_dir: str,
    raw_dir: str,
    test_size: float = 0.1,
    val_size: float = 0.1,
    test_seed: int = 0,
    val_seed: int = 0,
) -> None:
    """
    Performs group-aware split into training, validation, and testing sets and reorganizes data to set up for tensorflow.
    Creates data/raw/split/label/img file tree using data/grouped/group/label/img file tree.

    Args:
        grouped_dir (str): Directory with grouped images in grouped_dir/group/label/img structure.
        raw_dir (str): Location to save reorganized data in raw_dir/split/label/img structure.
        test_size (float): Percentage of samples to be used for testing.
        val_size (float): Percentage of samples to be used for validation.
        test_seed (int): Random seed to fix testing set group selection.
        val_seed (int): Random seed to fix validation set group selection (does not fix if test_seed changed).
    """
    # Identify groups and count how many samples they have
    groups_dict: dict[str, int] = {}
    for g in os.listdir(grouped_dir):
        if g.startswith('.'): continue  # ignore hidden files
        group_path = os.path.join(grouped_dir, g)
        if os.path.isdir(group_path):  # ignore non-folder files
            # Count all images inside all label folders for this group
            img_count: int = sum(
                len([f for f in os.listdir(os.path.join(group_path, label)) if not f.startswith('.')])
                for label in os.listdir(group_path)
                if not label.startswith('.') and os.path.isdir(os.path.join(group_path, label))
            )
            # Keep track of group names and their number of samples
            groups_dict[g] = img_count

    # Calculate how many samples should be in validation and testing splits
    total_samples: int = sum(groups_dict[g] for g in groups_dict)
    test_samples: int = int(total_samples * test_size)
    val_samples: int = int(total_samples * val_size)

    # Use monte carlo splitting to split by group while approximating split sample size
    test_groups, remaining_groups = _monte_carlo_split(groups_dict,
                                                       test_samples,
                                                       test_seed,
                                                       )
    remaining_groups_dict: dict[str, int] = {g: groups_dict[g] for g in remaining_groups}
    val_groups, train_groups = _monte_carlo_split(remaining_groups_dict,
                                                  val_samples,
                                                  val_seed,
                                                  )

    # Pair split names (for dir names) and groups lists
    splits: dict[str, list[str]] = {
        'test': test_groups,
        'val': val_groups,
        'train': train_groups,
    }
    print(f"""
    Groups in Training Set: {', '.join(train_groups)}.
    Groups in Validation Set: {', '.join(val_groups)}.
    Groups in Testing Set: {', '.join(test_groups)}.
    """)

    # Remove any existing train-test-val file tree
    # (we can't rely on overwriting if, for instance, we changed to test_size=0)
    if os.path.exists(raw_dir):
        shutil.rmtree(raw_dir)

    # Copy imgs from grouped/group/label/img tree to raw/split/label/img tree
    for split_name, group_list in splits.items():
        for group in group_list:
            group_path = os.path.join(grouped_dir, group)  # data/grouped/group

            for label in os.listdir(group_path):
                source_path = os.path.join(group_path, label)  # data/grouped/group/label
                # Skip any hidden folders or non-dir files
                if label.startswith('.') or not os.path.isdir(source_path):
                    continue
                dest_label: str = label.upper()  # make all labels uppercase in new file tree
                dest_path = os.path.join(raw_dir, split_name, dest_label)  # data/raw/split/label
                os.makedirs(dest_path, exist_ok=True)

                for image_file in os.listdir(source_path):
                    if image_file.startswith('.'): continue  # skip any hidden files

                    # Get absolute paths to ensure symlinks don't break
                    src_img = os.path.abspath(os.path.join(source_path, image_file)) # .../data/images/images/grouped/group/label/img
                    dest_img = os.path.abspath(os.path.join(dest_path, f"{group}_{dest_label}_{image_file}"))  # .../data/images/raw/split/label/img

                    # Skip hassle of symlinks with windows and just duplicate imgs instead
                    if os.name == 'nt':
                        shutil.copy2(src_img, dest_img)
                    else:
                        # Use symlinks to only store images once but in two locations
                        os.symlink(src_img, dest_img)

In [ ]:
GROUPED_DIR: str = "../data/images/grouped"
RAW_DIR: str = "../data/images/raw"

group_aware_split(
    grouped_dir=GROUPED_DIR,
    raw_dir=RAW_DIR,
    test_size=0.125,
    val_size=0.125,
    test_seed=42,  # <--- keep the same
    val_seed=42,  # <--- occassionally vary to check performance stability
    # Note: val_seed might have no influence for our small data set
)

After running the code above, your `data/` folder should look like this:
```
data/
  ├── grouped/         <--- initial grouped data
  │    ├── ben/        <--- each group gets their own folder
  │    │    ├── A/
  │    │    ├── B/
  │    │    └── C/
  │    ├── rockwell/
  │    │    ├── A/
  │    │    ├── B/
  │    │    └── C/
  │    ├── vasil/
  │    │    ├── A/
  │    │    ├── B/
  │    │    └── C/
  │    └── ...
  └── raw/             <--- created by group_aware_split()
       ├── train/      <--- data is now split into train, test, val instead of being organized by group
       │    ├── A/     <--- instead of duplicating the images, symlinks will refer to originals to save space (so don't delete grouped/!)
       │    ├── B/
       │    └── C/
       ├── test/
       │    ├── A/
       │    ├── B/
       │    └── C/
       └── val/
            ├── A/
            ├── B/
            └── C/
```

---
## 5. Processing Raw Images into Tensorflow Dataset Objects  📦

Now the data are organized properly into splits and ready for tensorflow to load with `image_dataset_from_directory`. We could stop here, but moving around images is not a lightweight and efficient approach. Instead, we can process those images once and save them in a more tensorflow-ready form which will make things significantly faster. The code below uses `image_dataset_from_directory` on each of the splits' folders (`raw/train/`, `raw/test/`, and `raw/val/`), and preprocesses them into this lightweight form. The processed data is then saved in a `data/processed/` folder.

In [ ]:
def process_raw_imgs(
    raw_dir: str,
    processed_dir: str,
) -> None:
    """
    Converts images into tensorflow.data.Dataset objects and saves them for fast loading.

    Args:
        raw_dir (str): Path to directory with raw images in raw_dir/split/label/img structure.
        processed_dir (str): Directory where processed data will be saved.
    """
    # Identify splits present in raw directory
    splits: list[str] = [
        s for s in os.listdir(raw_dir)
        if os.path.isdir(os.path.join(raw_dir, s))
    ]

    # Remove any existing processed file tree
    # (we can't rely on overwriting if, for instance, we changed to test_size=0)
    if os.path.exists(processed_dir):
        shutil.rmtree(processed_dir)

    for split in splits:
        # We only want the training set to shuffle
        # (shuffle=True changes the dataset object to reshuffle each time it is referenced which causes problems for things like confusion matrices later on)
        should_shuffle: bool = (split == "train")
        # Auto label images and convert to tensorflow.data.Dataset obejct
        ds: Dataset = image_dataset_from_directory(
            directory=os.path.join(raw_dir, split),
            labels='inferred',  # <--- inferring from directory structure
            label_mode='int',  # <--- encoding as 0, 1, 2. Means we should use sparse loss
            image_size=(256, 256),  # <--- reducing resolution to 256x256
            shuffle=should_shuffle,  # <--- only shuffle training set
            pad_to_aspect_ratio=True,  # <--- don't stretch image, just pad with 0s (black)
            batch_size=None,  # <--- we'll batch when loading so that we don't have to rerun this to change the batch size
        )
        # Save dataset
        dest_path = os.path.join(processed_dir, split)
        ds.save(dest_path)

In [ ]:
PROCESSED_DIR: str = "../data/images/processed"

process_raw_imgs(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
)

After running the code below, this is what the `data/` folder should look like:
```
data/
  ├── grouped/         <--- initial grouped data
  │    ├── ben/        <--- each group (student) gets their own folder
  │    │    ├── A/
  │    │    ├── B/
  │    │    └── C/
  │    ├── rockwell/
  │    │    ├── A/
  │    │    ├── B/
  │    │    └── C/
  │    ├── vasil/
  │    │    ├── A/
  │    │    ├── B/
  │    │    └── C/
  │    └── ...
  ├── raw              <--- created by group_aware_split()
  │    ├── train/      <--- data is now split into train, test, val instead of being organized by group
  │    │    ├── A/     <--- instead of duplicating the images, symlinks will refer to originals to save space (so don't delete grouped/!)
  │    │    ├── B/
  │    │    └── C/
  │    ├── test/
  │    │    ├── A/
  │    │    ├── B/
  │    │    └── C/
  │    └── val/
  │         ├── A/
  │         ├── B/
  │         └── C/
  └── processed        <--- created by process_raw_imgs()
       ├── train/      <--- data is no longer being stored as hi-res images, it is now much more lightweight
       ├── test/
       └── val/
```

---
## 6. Loading in Tensorflow Datasets  📫

If you're working locally, you won't have to do any of that again, the data is processed and you only need to work with the processed version going forwards (unless you want to change what is in the validation split). In future notebooks, you'll only need the code below this. In colab, this stuff doesn't get stored in persistently so you'll have to rerun it all each time the runtime gets deleted.

Once you have this processed data, you can load it into your notebook with the following:

In [ ]:
def load_datasets(
    processed_dir: str,
    batch_size: int = 32,
) -> dict[str, Dataset]:
    """
    Loads, batches, and prefetches processed data. Returns in dictionary with tf.data.Dataset objects for each split.

    Args:
        processed_dir (str): Path to directory with dir/split/Dataset structure
        batch_size (int): Batch size

    Returns:
        dict: mapping of split names to their respective tf.data.Dataset objects

    Examples:
    >>> datasets = load_datasets("../data/processed/", batch_size=32)
    >>> model.fit(
    ...     datasets['train'],
    ...     validation_data=datasets['val'],
    ...     epochs=10
    ... )
    """
    # Identify splits present in processed directory
    splits: list[str] = [
        s for s in os.listdir(processed_dir)
        if os.path.isdir(os.path.join(processed_dir, s))
    ]

    # Initialize autotune for prefetching
    autotune = tf.data.AUTOTUNE

    # Initialize dict that will be returned
    datasets: dict[str, Dataset] = {}

    for split in splits:
        # Load each dataset
        path = os.path.join(processed_dir, split)
        ds: Dataset = Dataset.load(path)
        # Batch, cache, and prefetch for optimal performance
        datasets[split]  = (
            ds
            .batch(batch_size)
            .cache()
            .prefetch(buffer_size=autotune)
        )

    return datasets

In [ ]:
# Load saved Dataset objects
datasets: dict[str, Dataset] = load_datasets(
    processed_dir=PROCESSED_DIR,
    batch_size=16, # <--- try changing this to influence training dynamics
)

Now we have our datasets split, processed, batched, and ready to use!

We can take a look at what our preprocessing did to the images. They have now been padded and their resolution has been lowered so that they can all be 256x256.

In [ ]:
import matplotlib.pyplot as plt

class_names: list[str] = ['A', 'B', 'C']

plt.figure(figsize=(10, 10))
for images, labels in datasets['train'].take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

Later on when we need the datasets we can access them through this dictionary like this:
```
model.fit(
    datasets['train'],
    validation_data=datasets['val'],
    ...
)
```

---
## 7.&nbsp; Creating the model 🤖

Now that our data is prepared, it's time to start building the model.

### 7.1. Train the Model

You'll see here that we've included the normalisation of the data as the first layer of the model. We've placed it here to keep everything together and not add an extra step. This model is fairly basic - when working with your images, it'll be up to you to use what you've learned so far to improve on this.

In [ ]:
num_classes = 3

model: Sequential = Sequential([
  Input(shape=(256, 256, 3)),
  Rescaling(1./255),
  Conv2D(32, 3, padding='same', activation='relu'),
  MaxPooling2D(),
  Conv2D(32, 3, padding='same', activation='relu'),
  MaxPooling2D(),
  Conv2D(32, 3, padding='same', activation='relu'),
  MaxPooling2D(),
  Flatten(),
  Dense(128, activation='relu'),
  Dense(num_classes, activation='softmax'),
])

In [ ]:
model.summary()

In [ ]:
model.compile(
  optimizer='adam',
  loss='sparse_categorical_crossentropy',
  metrics=['accuracy'],
)

In [ ]:
es = EarlyStopping(
    monitor="val_loss",
    patience=10,
    verbose=1,
    restore_best_weights=True,
)

In [ ]:
EPOCHS = 50

history: History = model.fit(
  datasets['train'],
  validation_data=datasets['val'],
  epochs=EPOCHS,
  #callbacks = [es],
  verbose = 1,
)

### 7.2. Training Curves

Let's have a look at the training curves to see how out model is performing.

In [ ]:
acc: list[float] = history.history['accuracy']
val_acc: list[float] = history.history['val_accuracy']
loss: list[float] = history.history['loss']
val_loss: list[float] = history.history['val_loss']
epochs_range: range = range(len(acc))

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(epochs_range, acc, label='Train')
plt.plot(epochs_range, val_acc, label='Val')
plt.title('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs_range, loss, label='Train')
plt.plot(epochs_range, val_loss, label='Val')
plt.title('Loss')
plt.legend()
plt.show()

---
### 7.3.&nbsp; Evaluate and Inspect Predictions 🔍

In [ ]:
val_loss, val_acc = model.evaluate(datasets['val'])
print(f"Validation accuracy: {val_acc:.2%}")

# Collect all images and labels from the validation dataset
all_images: list[Tensor] = []
all_labels: list[Tensor] = []
for images, labels in datasets['val']:
    all_images.append(images)
    all_labels.append(labels)

all_images: Tensor = concat(all_images, axis=0)
all_labels: Tensor = concat(all_labels, axis=0)

preds: np.ndarray = model.predict(all_images)
pred_labels: np.ndarray = np.argmax(preds, axis=1)

num_images: int = len(all_images)

# Calculate the number of rows and columns for the subplot grid
n_cols: int = 5  # You can adjust the number of columns as needed
n_rows: int = (num_images + n_cols - 1) // n_cols # Calculate rows needed

plt.figure(figsize=(n_cols * 2.5, n_rows * 3)) # Adjust figure size based on grid size
for i in range(num_images):
    plt.subplot(n_rows, n_cols, i + 1)
    plt.imshow(all_images[i].numpy().astype("uint8"))
    true_label: str = class_names[all_labels[i]]
    predicted_label: str = class_names[pred_labels[i]]
    plt.title(
        f"True: {true_label}\nPred: {predicted_label}",
        color='green' if true_label == predicted_label else 'red')
    plt.axis('off')
plt.tight_layout()
plt.show()

---
## 8. Challenges 👐

Now that you have the tools to perform a group-aware split, you're ready to do it yourself!
1. **Download the Data**: Download the datasets from the Google Drive and set up your local file tree.
2. **Perform the Split**: Copy the code above to perform the split.
3. **Train a Model to Identify Sign Language!** Try fitting a model to the training data and see how it performs on the testing set! You might find that this isn't as easy as it was with the MNIST data. Don't worry, over the next couple lessons we'll show you some techniques to help tackle this new challenge.

**Be Patient!**<br>
With a task like this, a model might seem like it's not improving for 10 or even 20 epochs before the loss suddently starts to drop. You'll definitely want to increase the patience if you want to continue using early stopping.

> **Note:** Training a CNN from scratch on this task can be slow. The model may show no improvement for 10-20 epochs before the loss suddenly drops. Be patient and use high EarlyStopping patience values (10+).

In the example above we can see that the loss wasn't noticably improving for the first ~15 epochs and accuracy was about as bad as it can be (33% with only 3 classes!). But then, after several minutes of seemingly dismal performance, accuracy suddenly started to climb! You may need to walk away from your computer for a few minutes while it's training to give it a chance to surpirse you.

---
## Summary

In this notebook, we:

1. **Organized our data** using directory structure for automatic labeling
2. **Implemented group-aware splitting** using Monte Carlo optimization to ensure no person appears in multiple splits
3. **Processed images** into efficient TensorFlow Dataset objects (256x256, padded)
4. **Trained a baseline CNN** from scratch as our performance benchmark

### Key Takeaway

A simple CNN trained from scratch can sometimes achieve decent accuracy, but performance is **unstable** and highly dependent on the random split. The model struggles to learn robust features from limited data. In the next notebook, we'll explore **data augmentation** to help with this, and in Notebook 3, we'll see how **transfer learning** dramatically improves results.

### Next Step

Proceed to **[Notebook 2: Image Augmentation](02_image_augmentation.ipynb)** to explore data augmentation techniques.